In [1]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import datetime

# 불필요한 경고문 생략(선택)
import warnings
warnings.filterwarnings('ignore')

# 모든 컬럼 출력설정(선택)
pd.set_option('display.max_columns', None)

#데이터 불러오기 
df = pd.read_csv('total_data.csv',index_col=0)

print('[행/컬럼 갯수]')
print(f"행: {df.shape[0]}, 컬럼: {df.shape[1]}\n")


[행/컬럼 갯수]
행: 51279, 컬럼: 40



In [2]:
#주차 컬럼 날짜타입 변환 (범주->날짜형)
df['주차'] = pd.to_datetime(df['주차'], format='%Y%m%d')

In [3]:
# 결측치 확인 -> 없음 
df.isna().sum()

기간        0
주차        0
라인        0
성별        0
기획년도      0
시즌이월      0
상품년차      0
시즌        0
복종        0
소품종       0
CAT       0
총입고수량     0
총입고원가     0
총입고택가     0
총출고수량     0
총출고원가     0
총출고택가     0
판매액       0
판매수량      0
매출원가      0
판매택가      0
총판매액      0
총판매수량     0
총매출원가     0
총판매택가     0
물류재고수량    0
물류재고원가    0
물류재고택가    0
매장재고수량    0
매장재고원가    0
매장재고택가    0
재고수량      0
재고원가      0
재고택가      0
기간입고수량    0
기간입고원가    0
기간입고택가    0
기간출고수량    0
기간출고원가    0
기간출고택가    0
dtype: int64

In [4]:
# 중복값 확인 및 제거 -> 전체 중복 12개
df.duplicated().sum() 
df.drop_duplicates(inplace=True)
# df[df.duplicated(keep=False)].sort_values(by='주차')

print('[행/컬럼 갯수임]')
print(f"행: {df.shape[0]}, 컬럼: {df.shape[1]}\n")

[행/컬럼 갯수임]
행: 51267, 컬럼: 40



# 범주형 컬럼 확인

In [5]:
#성별 소품종 클래스 확인 : 기타로 분류되는 클래스 2개 존재
#--> 최종 '기타' 로 오분류된 항목 52개 변환
display(df['성별'].value_counts())

filtered_df = df[df['성별'].str.contains('기타')]
filtered_df['소품종'].value_counts()

#봄 패딩 베스트만, '기타' 로 분류됨 -> 성별 구분 착오 예상 --> 봄패딩베스트 '1:남성' 값으로 변환 
con = (df['소품종']=='패딩베스트') & (df['시즌']=='봄')
df.loc[con,'성별'] = '1:남성'

df.loc[con,'성별'].value_counts()

성별
1:남성      47346
3:남녀공용     2510
2:여성       1322
4:기타         89
Name: count, dtype: int64

성별
1:남성    52
Name: count, dtype: int64

- (추가설명) 컬럼 중에 시즌 이월이라는 컬럼 존재 
    - 제품별로 시즌이 있는데 그거 기준으로 시즌별 마감일자가 존재함
    - 같은 상품이어도 해당 연도에 봄상품이 일정 날짜를 지나면 이월로 취급함
    - 모델링할때는 이월제품 제거하고 해도 되겠다 : 판매가 끝났기 때문에 할인율 변동이 필요가 없을 듯
    - 시즌 마감일을 넘기면 할인율을 멕일 이유가 없음

In [6]:
#시즌이월 컬럼 클래스 확인 : 이월제품 의미 파악 필요
##결론 : 판매예측/할인최적화 모델링시 '이월' 행 삭제 ( -12578 32%) ,년간 매출 집계시 유지
display(df['시즌이월'].value_counts())

#2024년도 기준 시즌/이월 여부 확인 
df_2024 = df[df['기획년도']==2024]
df_2024 = df_2024.drop(columns=['기간','상품년차'])

# # 범주형 최소단위 필터링을 위한 '카테고리' 컬럼 생성
df_2024['카테고리'] = df_2024['시즌'] + "_" +  df_2024['복종'] + "_" + df_2024['소품종'] + "_" + df_2024['라인']+ "_" + df_2024['성별']

# #카테고리별 시즌/이월값이 둘다 있는거 소팅 -> '샘플 가을_니트 셔츠_라운드_ZB_1:남성'   확인
df_2024.groupby('카테고리')['시즌이월'].nunique()

# #샘플확인 : '가을_니트 셔츠_라운드_ZB_1:남성' -> 시즌별 마감 이후 이월로 변경 됨 
con = df_2024['카테고리'] == '가을_니트 셔츠_라운드_ZB_1:남성'
smpl = df_2024[con].sort_values(by='주차',ascending=True)
smpl[(smpl['주차'] >'2024-11-01') & (smpl['주차'] <='2024-12-30')]

# 시즌-> 이월 바뀌는 시점 함수화 : gpt
def find_transition_points(group):
    group = group.sort_values('주차')
    transition_rows = group[(group['시즌이월'].shift(1) == '01_시즌') & (group['시즌이월'] == '02_이월')]
    return transition_rows[['카테고리', '주차']]

transition_points = df_2024.groupby('카테고리', group_keys=False).apply(find_transition_points)

# 시즌 -> 이월로 바뀌는 주차만 출력
transition_points['주차'].unique()


시즌이월
01_시즌    38689
02_이월    12578
Name: count, dtype: int64

<DatetimeArray>
['2024-12-01 00:00:00', '2024-06-02 00:00:00', '2024-10-06 00:00:00']
Length: 3, dtype: datetime64[ns]

# 수치형 컬럼 & 집계 컬럼 점검
사용 컬럼 : '총입고수량','총입고원가','총입고택가','총출고수량','총출고원가','총출고택가','판매수량','판매액','매출원가','판매택가','총판매액','총판매수량','총매출원가','총판매택가','재고수량','재고원가','재고택가'

In [7]:
# 파생변수 생성
#1. 범주형 최소단위 필터링을 위한 '카테고리' 컬럼 생성
df['카테고리'] = df['시즌'] + "_" +  df['복종'] + "_" + df['소품종'] + "_" + df['라인']
num_df = df[['카테고리','주차','총입고수량','총입고원가','총입고택가','판매수량','판매액','매출원가','판매택가','총판매액','총판매수량','총매출원가','총판매택가','재고수량','재고원가','재고택가','시즌','복종','소품종','라인','시즌이월']]

#2. 입고기준 원가/택가 
num_df['제품원가'] = (num_df['총입고원가'] / num_df['총입고수량'])
num_df['제품택가'] = (num_df['총입고택가'] / num_df['총입고수량'])

- (추가설명) 입고 되지 않은 상품이 판매되지 않음 (입고수량은 비교적 명확하다)
    - 질문 : 입고 수량이 작년에 입고됐지만 지금은 입고 되지 않은 경우는? 총 입고 수량은 입고라 한번 입고 되면 무조건 같이 찍혀있어야 함 / 전산의 이유로 품번을 먼저 등록했다거나 등 부가적인 이유가 있을 수 있음

In [8]:
# 총입고수량/입고원가/입고택가 확인 
# 입고전 데이터 확인 및 행 삭제 : 162개 -> 입고되지 않은 상품은 출고 및 판매 불가, 예약판매 등 특수한 케이스 없다고 가정 
print((num_df['총입고수량'] == 0).sum())
filtered_df = num_df[(num_df['총입고수량'] > 0)]

print('[행/컬럼 갯수]')
print(f"행: {filtered_df.shape[0]}, 컬럼: {filtered_df.shape[1]}\n")


162
[행/컬럼 갯수]
행: 51105, 컬럼: 23



- (추가설명) 재고 관련 집계보다 다른 집계 컬럼들이 명확하기 때문에 명확한 데이터들로 다시 계산하는 게 확실하겠다.

In [9]:
# 재고수량/재고원가/재고택가 정합성 확인
# 결론: 재고관련 집계 컬럼 삭제 후 입고-판매 기준 다시 집계 (입고,판매 데이터의 신뢰도가 더 높다고 봄, 실제 wms랑 비교 할 수 없으므로 가정)
filtered_df['재고잔량_check'] = (filtered_df['총입고수량'] - filtered_df['총판매수량'] == filtered_df['재고수량'])
filtered_df['재고원가_check'] = (filtered_df['총입고원가'] - filtered_df['총매출원가'] == filtered_df['재고원가'])
filtered_df['재고택가_check'] = (filtered_df['총입고택가'] - filtered_df['총판매택가'] == filtered_df['재고원가'])

# 입고 - 판매 = 재고 안맞는 행 : 27569행
filtered_df[filtered_df[['재고잔량_check', '재고원가_check', '재고택가_check']].any(axis=1) == False]
(filtered_df[['재고잔량_check', '재고원가_check', '재고택가_check']].any(axis=1) == False).sum() #27569행

# 재고관련 컬럼 삭제
filtered_df.drop(columns=['재고수량','재고원가','재고택가','재고잔량_check','재고원가_check','재고택가_check'],inplace=True)

print('[행/컬럼 갯수]')
print(f"행: {filtered_df.shape[0]}, 컬럼: {filtered_df.shape[1]}\n")


[행/컬럼 갯수]
행: 51105, 컬럼: 20



# 판매수량/판매액/매출원가/판매택가 확인
- 1. 판매수량 0 , 집계컬럼(판매액/매출원가/판매택가) !=0 -> 이상치, 집계컬럼 값 0으로 변경
- 2. 판매택가 정합성 확인 (입고택가/입고수량 = 판매택가/판매수량) -> ok 
- 3. 판매수량 >0 , 집계컬럼 <=0
- 4. 판매수량 <0 , 집계컬럼 >=0
- 5. 판매수량 <0 , 집계컬럼 < 0 -> 정합성 안맞는거 확인 

In [10]:
# 음수 데이터 확인 -> 판매수량/판매액의 음수값 갯수가 다 다름
minus_con = filtered_df.select_dtypes(include='number') <0
minus_con.sum()

총입고수량       0
총입고원가       0
총입고택가       0
판매수량     1795
판매액      1508
매출원가     1863
판매택가     1795
총판매액        2
총판매수량       3
총매출원가       3
총판매택가       3
제품원가        0
제품택가        0
dtype: int64

In [11]:
# 1. 판매수량은 0인데, 판매액/매출원가가 있는 경우 -> 348개 
#결론: 판매 없이 판매액/매출원가/판매택가 발생할수없다고 판단, 이상치로 간주하고 매출원가/판매택가 0으로 변경 ==> 총 집계값도 수정해야함
con1 = filtered_df['판매수량'] == 0
con2 = filtered_df['판매액'] != 0
con3 = filtered_df['매출원가'] != 0

print((con1 & (con2 | con3)).sum())

#수량이 0일떄, 판매액/매출원가 0으로 변경
filtered_df.loc[(con1 & (con2 | con3)),['판매수량','판매액','매출원가']]=0

filtered_df[con1].describe()

348


,주차,총입고수량,총입고원가,총입고택가,판매수량,판매액,매출원가,판매택가,총판매액,총판매수량,총매출원가,총판매택가,제품원가,제품택가
count,7818,7818.000000,7.818000e+03,7.818000e+03,7818.0,7818.0,7818.0,7818.0,7.818000e+03,7818.000000,7.818000e+03,7.818000e+03,7818.000000,7.818000e+03
mean,2023-04-12 22:45:57.329240064,5090.849706,6.139490e+07,5.447658e+08,0.0,0.0,0.0,0.0,1.000217e+08,2900.524942,3.584569e+07,3.112481e+08,22411.038014,1.532622e+05
min,2021-01-03 00:00:00,1.000000,4.391500e+04,1.600000e+05,0.0,0.0,0.0,0.0,-9.900000e+04,-1.000000,-1.493000e+04,-9.900000e+04,166.000000,3.300000e+03
25%,2022-04-10 00:00:00,1000.000000,1.585521e+07,1.297400e+08,0.0,0.0,0.0,0.0,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,8214.479853,7.990000e+04
50%,2023-05-14 00:00:00,2476.000000,3.742508e+07,3.012170e+08,0.0,0.0,0.0,0.0,5.803200e+07,1198.000000,1.942430e+07,1.597523e+08,13593.708539,9.990000e+04
75%,2024-03-10 00:00:00,4992.000000,7.627107e+07,6.983010e+08,0.0,0.0,0.0,0.0,1.230972e+08,3017.000000,4.443482e+07,3.972910e+08,26914.839740,1.990000e+05
max,2024-12-29 00:00:00,182459.000000,7.369345e+08,9.104704e+09,0.0,0.0,0.0,0.0,2.023863e+09,106352.000000,5.749144e+08,5.306965e+09,277921.000000,1.299000e+06
std,NaN,8979.126126,7.666198e+07,6.934383e+08,0.0,0.0,0.0,0.0,1.636087e+08,6396.793839,6.038977e+07,5.022997e+08,26432.083484,1.255064e+05


In [12]:
# 2. 판매데이터 정합성 확인 : 개당 입고택가 와 개당 판매택가 일치 여부 확인 (원가는 입고/판매 원가 다르므로 제외 (클레임 등 ))
#결론 : 판매택가 정합성 ok 
con = (filtered_df['판매수량'] * filtered_df['제품택가'] != filtered_df['판매택가'])
filtered_df[con]

,카테고리,주차,총입고수량,총입고원가,총입고택가,판매수량,판매액,매출원가,판매택가,총판매액,총판매수량,총매출원가,총판매택가,시즌,복종,소품종,라인,시즌이월,제품원가,제품택가


In [13]:
# 3-1. 판매수량 >0 , 집계컬럼 <=0 (매출원가)
# 결론 : 4행 존재 집계 오류 판단 -> 입고원가 * 판매수량 으로 대치

con = (filtered_df['판매수량'] > 0) & (filtered_df['매출원가'] < 0)
display(con.sum())

filtered_df.loc[con,'매출원가']= filtered_df['제품원가'].round(0) * filtered_df['판매수량'] 
filtered_df[con]

# 3-2. 판매수량 <0 , 집계컬럼 >=0 (매출원가)
# 결론 : 3행 존재 집계 오류 판단 -> 입고원가 * 판매수량 으로 대치
con = (filtered_df['판매수량'] < 0) & (filtered_df['매출원가'] >= 0)

filtered_df.loc[con,'매출원가']= filtered_df['제품원가'].round(0) * filtered_df['판매수량'] 
filtered_df[con]

4

,카테고리,주차,총입고수량,총입고원가,총입고택가,판매수량,판매액,매출원가,판매택가,총판매액,총판매수량,총매출원가,총판매택가,시즌,복종,소품종,라인,시즌이월,제품원가,제품택가
11279,여름_니트 셔츠_라운드_ZB,2021-12-19,95434,289446297,2767586000,-1,0,-3033,-29000,452178679,48430,146928382,1404470000,여름,니트 셔츠,라운드,ZB,02_이월,3032.947346,29000.0
15469,봄_코트_싱글코트_ZB,2022-06-12,4485,135692109,1475565000,-1,-59000,-30255,-329000,346254799,2336,73110572,768544000,봄,코트,싱글코트,ZB,02_이월,30254.650836,329000.0
22999,여름_팬츠_니트팬츠_ZE,2022-12-04,10152,52045783,709624800,-1,-21920,-5127,-69900,65380713,4024,20810904,281277600,여름,팬츠,니트팬츠,ZE,02_이월,5126.653172,69900.0


In [14]:
# 판매수량 - 매출원가/판매택가 전처리 완료
minus_con = filtered_df.select_dtypes(include='number') <0
minus_con.sum()

총입고수량       0
총입고원가       0
총입고택가       0
판매수량     1795
판매액      1417
매출원가     1795
판매택가     1795
총판매액        2
총판매수량       3
총매출원가       3
총판매택가       3
제품원가        0
제품택가        0
dtype: int64

In [15]:

# 4-1. 판매수량 >0 , 집계컬럼 <0 (판매액)
#총 117행 -> 
con = (filtered_df['판매수량'] > 0) & (filtered_df['판매액'] < 0)
display(con.sum())
filtered_df[con].sort_values(by='판매수량',ascending=False)

# 4-2. 판매수량 <0 , 집계컬럼 >0 (판매액)
#총 257행 -> 
plus_con = (filtered_df['판매수량'] < 0) & (filtered_df['판매액'] > 0)
display(plus_con.sum())
filtered_df[con].sort_values(by='판매수량',ascending=False)

#고찰 : 판매액 연산 오류 ? or 판매수량이 오류인건지
# 위에 이상치 -> (추정 실판가) 판매액/판매수량 과 직전 실판가 비교  / 판매수량 과 직전 판매수량 비교  -> 집계 후 비교 진행 

117

257

,카테고리,주차,총입고수량,총입고원가,총입고택가,판매수량,판매액,매출원가,판매택가,총판매액,총판매수량,총매출원가,총판매택가,시즌,복종,소품종,라인,시즌이월,제품원가,제품택가
2827,여름_니트 셔츠_라운드_ZB,2021-05-16,19659,151434476,1374164100,197,-2065510,1517503,13770300,37861426,1395,10745770,97510500,여름,니트 셔츠,라운드,ZB,01_시즌,7703.060990,69900.0
16051,봄_팬츠_팬츠(일반)_ZE,2022-07-03,19312,171369250,1929268800,153,-742495,1361790,15284700,406820415,15697,138852015,1568130300,봄,팬츠,팬츠(일반),ZE,02_이월,8873.718413,99900.0
10646,겨울_니트 셔츠_라운드_ZE,2021-12-05,25000,162488643,1747500000,147,-259841,955434,10275300,408557391,21665,140812691,1514383500,겨울,니트 셔츠,라운드,ZE,01_시즌,6499.545720,69900.0
3127,여름_팬츠_반바지_ZB,2021-05-30,15805,94078465,788669500,123,-889853,737933,6137700,17413041,642,3837893,32035800,여름,팬츠,반바지,ZB,01_시즌,5952.449541,49900.0
5920,봄_니트 셔츠_라운드_ZB,2021-08-22,30496,192806824,2131670400,108,-501187,684297,7549200,320035813,14612,92357678,1021378800,봄,니트 셔츠,라운드,ZB,02_이월,6322.364376,69900.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26020,여름_팬츠_팬츠(일반)_ZA,2023-04-09,1361,25567379,189179000,1,-361400,18771,139000,10044300,102,1915979,14178000,여름,팬츠,팬츠(일반),ZA,01_시즌,18785.730345,139000.0
28467,여름_스웨터_라운드_ZF,2023-06-25,296,4034483,29570400,1,-1330,13630,99900,3202231,62,845061,6193800,여름,스웨터,라운드,ZF,01_시즌,13630.010135,99900.0
380,봄_수트_수트팬츠_ZB,2021-01-31,1820,37928190,180180000,1,-108900,20840,99000,7899800,109,2271523,10791000,봄,수트,수트팬츠,ZB,01_시즌,20839.664835,99000.0
3876,봄_수트_수트팬츠_ZB,2021-06-20,1820,37928190,180180000,1,-15190,20840,99000,66397799,1172,24424087,116028000,봄,수트,수트팬츠,ZB,02_이월,20839.664835,99000.0


In [16]:
# 1. 추정 실판가(판매액/판매수량) 과 직전 실판가 비교  / 추정 판매수량 과 직전 판매수량 비교
con1 = (filtered_df['판매수량'] > 0) & (filtered_df['판매액'] < 0)
con2 = (filtered_df['판매수량'] < 0) & (filtered_df['판매액'] > 0)
display((con1 | con2).sum())

# 1. 데이터 정렬 (카테고리, 총입고수량, 주차 기준)
filtered_df = filtered_df.sort_values(by=['카테고리','총입고수량','주차'])

# 2-1. 직전 주차의 실판가 계산 & 직전 주차 판매수량
filtered_df['직전주차_판매수량'] = filtered_df.groupby(['카테고리','총입고수량'])['판매수량'].shift(1)
filtered_df['직전주차_판매액'] = filtered_df.groupby(['카테고리','총입고수량'])['판매액'].shift(1)

filtered_df['직전주차_실판가'] = (filtered_df['직전주차_판매액'] / filtered_df['직전주차_판매수량']).round(0)
filtered_df['추정_실판가'] = (filtered_df['판매액']/ filtered_df['판매수량']).round(0)
# filtered_df['추정_실판가'] = filtered_df['추정_실판가'].fillna(0)  # NaN 값 0으로 대체
filtered_df.loc[(con1|con2),['카테고리','주차','판매수량','판매액','직전주차_판매수량','추정_실판가','직전주차_실판가','매출원가','판매택가']]

# 엑셀로 점검 
a = filtered_df.loc[(con1|con2),['카테고리','주차','판매수량','판매액','직전주차_판매수량','추정_실판가','직전주차_실판가','매출원가','판매택가']]
a
# a.to_excel('check_sales.xlsx')


374

,카테고리,주차,판매수량,판매액,직전주차_판매수량,추정_실판가,직전주차_실판가,매출원가,판매택가
4776,가을_수트_수트팬츠_ZB,2021-07-18,5,-1304808,83.0,-260962.0,127516.0,96220,645000
45985,가을_스웨터_T-에리_ZA,2024-09-29,25,-115000,127.0,-4600.0,52905.0,378755,2725000
47788,가을_스웨터_T-에리_ZA,2024-11-03,-6,273000,6.0,-45500.0,39000.0,-90901,-654000
47805,가을_스웨터_T-에리_ZB,2024-11-03,-23,156000,9.0,-6783.0,39000.0,-434478,-2047000
5258,가을_우븐 셔츠_드레스셔츠_ZE,2021-08-01,-2,170676,166.0,-85338.0,19970.0,-10921,-159800
...,...,...,...,...,...,...,...,...,...
19507,여름_팬츠_팬츠(일반)_ZE,2022-10-02,-5,134791,32.0,-26958.0,19507.0,-46270,-499500
21583,여름_팬츠_팬츠(일반)_ZE,2022-11-06,-271,164370,3.0,-607.0,14872.0,-2508480,-27072900
7599,여름_팬츠_팬츠(일반)_ZE,2021-10-03,-4,75850,0.0,-18962.0,NaN,-39683,-399600
30978,여름_팬츠_팬츠(일반)_ZF,2023-09-03,1,-19564,9.0,-19564.0,28296.0,17840,139000


In [17]:
# 4-3. 판매수량 !=0 , 집계컬럼 ==0 (판매액) 
#총 284행 -> 금중 186행 직전실판가 0이하 or null  
con = (filtered_df['판매수량'] != 0) & (filtered_df['판매액'] == 0)
display(con.sum())
filtered_df[con].sort_values(by='판매수량',ascending=False).head(5)

# b.to_excel('check_sales2.xlsx')

284

,카테고리,주차,총입고수량,총입고원가,총입고택가,판매수량,판매액,매출원가,판매택가,총판매액,총판매수량,총매출원가,총판매택가,시즌,복종,소품종,라인,시즌이월,제품원가,제품택가,직전주차_판매수량,직전주차_판매액,직전주차_실판가,추정_실판가
21335,여름_팬츠_반바지_ZE,2022-11-06,12397,89419608,990520300,334,0,2409143,26686600,150318410,11455,82624958,915254500,여름,팬츠,반바지,ZE,02_이월,7213.003791,79900.0,0.0,0.0,NaN,0.0
21704,여름_우븐 셔츠_캐쥬얼셔츠_ZE,2022-11-06,18555,118298313,1853644500,161,0,1026463,16083900,251938852,15171,96723455,1515582900,여름,우븐 셔츠,캐쥬얼셔츠,ZE,02_이월,6375.549070,99900.0,0.0,0.0,NaN,0.0
21356,여름_니트 셔츠_라운드_ZF,2022-11-06,28848,88747205,2016475200,153,0,470685,10694700,57879728,4659,14332821,325664100,여름,니트 셔츠,라운드,ZF,02_이월,3076.372885,69900.0,16.0,160000.0,10000.0,0.0
21617,여름_니트 셔츠_라운드_ZE,2022-11-06,11953,79930442,835514700,143,0,956787,9995700,118092166,7318,48919434,511528200,여름,니트 셔츠,라운드,ZE,02_이월,6687.061156,69900.0,0.0,0.0,NaN,0.0
21574,여름_팬츠_반바지_ZF,2022-11-06,3588,25397178,286681200,125,0,884796,9987500,25410434,1741,12323436,139105900,여름,팬츠,반바지,ZF,02_이월,7078.366221,79900.0,0.0,0.0,NaN,0.0


# 1차 결론
- 판매데이터 오류 (ex. 판매수량은 양수, 판매액은 음수,판매액은0 판매수량은 존재 등
- 판매수량 오류보다는 판매액집계 오류 가능성이 더 커보임 => '직전 주차' or '평균 실판가'로 대치 필요
- 1. 직전주차 실판가 로 대치시킨다면, 0이하이거나 null인 행 57개 추가로 어떻게 대치 시킬지 논의 필요

# 시계열 , 할인 모델링 전처리 진행
-라인 제거/ 소품종 제거
-시즌이월 '이월' 제거
- 

In [18]:
# 특정 라인(ZD, ZE, ZF) 제거
filtered_df = filtered_df[~filtered_df['라인'].isin(['ZD', 'ZE', 'ZF'])]

# '소품', '언더웨어' 제거
filtered_df = filtered_df[~filtered_df['복종'].isin(['소품', '언더웨어'])]

# 시즌이월 '이월' 제거
filtered_df = filtered_df[~filtered_df['시즌이월'].isin(['02_이월'])]

print('[행/컬럼 갯수]')
print(f"행: {filtered_df.shape[0]}, 컬럼: {filtered_df.shape[1]}\n")
filtered_df.head(5)

[행/컬럼 갯수]
행: 24074, 컬럼: 24



,카테고리,주차,총입고수량,총입고원가,총입고택가,판매수량,판매액,매출원가,판매택가,총판매액,총판매수량,총매출원가,총판매택가,시즌,복종,소품종,라인,시즌이월,제품원가,제품택가,직전주차_판매수량,직전주차_판매액,직전주차_실판가,추정_실판가
44454,가을_니트 셔츠_라운드_ZB,2024-08-25,5037,29660977,397923000,82,1898000,482867,6478000,1898000,82,482867,6478000,가을,니트 셔츠,라운드,ZB,01_시즌,5888.619615,79000.0,NaN,NaN,NaN,23146.0
44522,가을_니트 셔츠_라운드_ZB,2024-09-01,5037,35081791,397923000,58,772000,342616,4582000,2670000,140,975075,11060000,가을,니트 셔츠,라운드,ZB,01_시즌,6964.818543,79000.0,82.0,1898000.0,23146.0,13310.0
44979,가을_니트 셔츠_라운드_ZB,2024-09-08,5037,35081791,397923000,57,1083000,396995,4503000,3753000,197,1372069,15563000,가을,니트 셔츠,라운드,ZB,01_시즌,6964.818543,79000.0,58.0,772000.0,13310.0,19000.0
45275,가을_니트 셔츠_라운드_ZB,2024-09-15,5037,35081791,397923000,81,1539000,564150,6399000,5292000,278,1936220,21962000,가을,니트 셔츠,라운드,ZB,01_시즌,6964.818543,79000.0,57.0,1083000.0,19000.0,19000.0
45859,가을_니트 셔츠_라운드_ZB,2024-09-22,5037,35081791,397923000,198,3762000,1379034,15642000,9054000,476,3315254,37604000,가을,니트 셔츠,라운드,ZB,01_시즌,6964.818543,79000.0,81.0,1539000.0,19000.0,19000.0


In [19]:
# 그룹화 및 집계 연산 적용 :최종 컬럼 8867행
group_cols = ['카테고리','주차']
agg_dict = {
    '총입고수량': 'sum',
    '총입고원가': 'sum',
    '총입고택가': 'sum',

    '판매수량': 'sum',
    '판매액': 'sum',
    '매출원가': 'sum',
    '판매택가': 'sum',
    
    '제품원가' : 'mean',
    '제품택가' : 'mean'
}
filtered_df = filtered_df.groupby(group_cols).agg(agg_dict).reset_index()

In [ ]:
#파생변수 

# 실판가 계산 (판매수량/판매액 0일때 :238개 )
filtered_df['실판가'] = np.where(
    (filtered_df['판매액'] == 0) | (filtered_df['판매수량'] == 0), 
    0, 
    filtered_df['판매액'] / filtered_df['판매수량']
)

# 주간할인율 계산 (판매택가가 0이면 1로 일단 출력 - 직전 할인율로 대치 )
filtered_df['주간할인율'] = np.where(
    filtered_df['판매택가'] == 0,
    1,
    (filtered_df['판매택가'] - filtered_df['판매액']) / filtered_df['판매택가'] * 100,
)

# filtered_df['원가판매율'] = filtered_df['총매출원가']/filtered_df['총입고원가']*100
# filtered_df['수량판매율'] = filtered_df['총판매수량']/filtered_df['총입고수량']*100


filtered_df
# filtered_df.to_excel('tableau_check.xlsx')

# 필요시 여기까지 가공된, filtered_df 파일 csv 로 시계열 추이 확인 가능 

In [21]:
# 판매수량 - 매출원가/판매택가 확인 
minus_con = filtered_df.select_dtypes(include='number') <0
minus_con.sum()

총입고수량      0
총입고원가      0
총입고택가      0
판매수량     121
판매액      149
매출원가     129
판매택가     124
제품원가       0
제품택가       0
실판가       31
주간할인율     38
dtype: int64

# 판매수량 음수값 기준 이상치 확인
- 1. 1.5 iqr
- 2. 3시그마
- 3. 매장기준 -200 절대값


In [22]:
# 판매수량이 음수인 값만 필터링
negative_sales_df = filtered_df[filtered_df['판매수량'] < 0]

# IQR 계산
Q1 = negative_sales_df['판매수량'].quantile(0.25)
Q3 = negative_sales_df['판매수량'].quantile(0.75)
IQR = Q3 - Q1

# 3 * IQR 기준으로 이상치 판별
lower_bound = Q1 - 1.5 * IQR

# 이상치 개수 확인
outliers_count = (negative_sales_df['판매수량'] < lower_bound).sum()

# 결과 출력
print(f"이상치 기점: {lower_bound}, 3 IQR 이상인 이상치 개수: {outliers_count}")

이상치 기점: -246.0, 3 IQR 이상인 이상치 개수: 20


In [23]:
# 3표준편차 -> 하한 지점이 너무 낮음 -> 탈락 
mean = negative_sales_df['판매수량'].mean()
stddev = negative_sales_df['판매수량'].std()
lower_bound2 = mean - 3 * stddev  # 3표준편차 하한

# 이상치 개수 확인
outliers_count2 = (negative_sales_df['판매수량'] < lower_bound2).sum()

# 결과 출력
print(f"이상치 기점: {lower_bound2}, 3시그마 이상인 이상치 개수: {outliers_count2}")

이상치 기점: -1157.0053274530037, 3시그마 이상인 이상치 개수: 4


In [24]:
# 전국 매장 수 기준 : 약 200개  
# 이상치 개수 확인
outliers_count3 = (negative_sales_df['판매수량'] < -200).sum()

# 결과 출력
print(f"이상치 기점: -200, 3 IQR 이상인 이상치 개수: {outliers_count3}")

이상치 기점: -200, 3 IQR 이상인 이상치 개수: 21


In [25]:
# 카테고리별 개수 세기 - 1.5 iqr 기준
category_counts = negative_sales_df[negative_sales_df['판매수량'] < lower_bound]
category_outlier_counts = category_counts['카테고리'].value_counts()
category_outlier_counts

카테고리
사계절_니트 셔츠_라운드_ZB      7
사계절_데님_데님팬츠_ZB        6
여름_니트 셔츠_라운드_ZB       2
여름_자켓_싱글재킷_ZB         2
겨울_스웨터_라운드_ZB         1
봄_자켓_싱글재킷_ZB          1
사계절_우븐 셔츠_드레스셔츠_ZB    1
Name: count, dtype: int64